# Notebook 4: Muestreo de Imágenes Ópticas de Alta Resolución (Sentinel-2)
### Proyecto GeoVision-CLIP Cali | Componente de Visión Artificial

Este notebook documenta la extracción de imágenes multiespectrales que servirán de base para el entrenamiento de la red neuronal **CLIP**. Estas imágenes permiten identificar la infraestructura urbana y la cobertura vegetal que influyen en la dispersión de contaminantes.

## 1. Misión y Sensor (Contexto Técnico)
* **Misión:** [Copernicus Sentinel-2](https://sentinels.copernicus.eu/web/sentinel/missions/sentinel-2), operada por la Agencia Espacial Europea (ESA).
* **Sensor:** **MSI** (Multi-Spectral Instrument).
* **Resolución Espacial:**
  * **10 metros:** Bandas visibles (R, G, B) y Near-Infrared (NIR).
  * **20 metros:** Bandas de "Borde Rojo" (Red Edge) y SWIR.
* **Frecuencia:** Revisit de 5 días sobre Cali, permitiendo un monitoreo dinámico del crecimiento urbano.

## 2. Formato y Procesamiento de Datos
Trabajaremos con el dataset **S2_SR_HARMONIZED** (Surface Reflectance):
* **Nivel de Procesamiento:** Nivel-2A (Corregido atmosféricamente). Los valores representan la reflectancia real en la superficie, eliminando el "ruido" de la atmósfera.
* **Filtro de Calidad:** Dado que Cali es una zona tropical, aplicamos un filtro estricto de **nubosidad < 60%** para garantizar que la IA no aprenda patrones de nubes en lugar de patrones urbanos.
* **Unidades:** Valores de reflectancia escalados (0 a 10,000).

## 3. Utilidad en GeoVision-CLIP
Estas imágenes serán procesadas para extraer texturas industriales y densidades de vegetación (NDVI). La red neuronal CLIP aprenderá a asociar, por ejemplo, los techos metálicos de Acopi-Yumbo con los picos de $SO_2$ observados en los notebooks anteriores.

In [ ]:
import ee
import geemap

# 1. Rutina de Autenticación Segura para el equipo
try:
    # Intenta inicializar directamente (funciona si ya te autenticaste antes)
    ee.Initialize(project='proyecto3ia-494900')
    print("Conectado a Google Earth Engine exitosamente.")
except Exception as e:
    print(" Autenticación requerida. Sigue las instrucciones del navegador...")
    ee.Authenticate() # Lanza el pop-up para el compañero que entra por primera vez
    ee.Initialize(project='proyecto3ia-494900')
    print("Autenticación completada y conectado a GEE.")

## 4. Visualización de Muestreo: Color Real vs. Infrarrojo
Para este muestreo, utilizaremos una técnica de **Composición de Mediana**. Tomamos todas las fotos de Cali entre 2018 y 2022 y elegimos el valor mediano de cada píxel. Esto crea un "mosaico perfecto" libre de nubes y sombras.

Visualizaremos dos combinaciones:
1. **Color Real (TCI):** Bandas B4(R), B3(G), B2(B). Es como lo vería el ojo humano.
2. **Falso Color Infrarrojo:** Bandas B8(NIR), B4(R), B3(G). Aquí, la vegetación se ve roja intensa, permitiendo diferenciar parques y zonas agrícolas de la zona industrial gris.

In [4]:
# 1. Definir ROI (Cali) y Fechas
cali_roi = ee.Geometry.Rectangle([-76.60, 3.30, -76.40, 3.55])
fecha_inicio = '2018-01-01'
fecha_fin = '2022-12-31'

# 2. Cargar Colección Sentinel-2 (Filtrada por nubes < 60%)
s2_coleccion = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                .filterBounds(cali_roi)
                .filterDate(fecha_inicio, fecha_fin)
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60)))

# 3. Crear Mosaico de Mediana (Limpio de nubes)
s2_mosaico = s2_coleccion.median().clip(cali_roi)

# 4. Configurar Parámetros de Visualización
viz_color_real = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0.0,
    'max': 3000.0,
    'gamma': 1.2
}

viz_falso_color = {
    'bands': ['B8', 'B4', 'B3'],
    'min': 0.0,
    'max': 5000.0,
    'gamma': 1.1
}

# 5. Renderizar Mapa
Map = geemap.Map(basemap='SATELLITE')
Map.centerObject(cali_roi, 12)

# Añadimos la imagen NORMAL (Color Real) visible por defecto
Map.addLayer(s2_mosaico, viz_color_real, 'Sentinel-2 Color Real (RGB)')

# Añadimos la imagen ROJA (Infrarrojo), pero AGREGAMOS shown=False para que inicie apagada
Map.addLayer(s2_mosaico, viz_falso_color, 'Sentinel-2 Infrarrojo (Vegetación)', shown=False)

Map.addLayerControl()
print(f"Se procesaron {s2_coleccion.size().getInfo()} imágenes para crear este mosaico.")
Map

Se procesaron 204 imágenes para crear este mosaico.


Map(center=[3.424999763692953, -76.50000000000118], controls=(WidgetControl(options=['position', 'transparent_…

## 5. Extracción de Características: Índice de Vegetación (NDVI)
Para que nuestro modelo **GeoVision-CLIP** pueda correlacionar la infraestructura con la calidad del aire, necesitamos cuantificar la cobertura vegetal. Las zonas de asfalto y techos industriales (bajo NDVI) tienden a ser "islas de calor" que atrapan contaminantes, mientras que los parques y zonas rurales (alto NDVI) facilitan la dispersión y absorción.

### Fundamento Matemático
El NDVI aprovecha la forma en que las plantas reflejan la luz. La clorofila absorbe la luz visible (Roja) para la fotosíntesis, pero refleja fuertemente el Infrarrojo Cercano (NIR). La fórmula aplicada a los tensores es:

$$NDVI = \frac{NIR - Red}{NIR + Red}$$

En la arquitectura de **Sentinel-2**, esto se traduce matemáticamente a las bandas 8 y 4:

$$NDVI = \frac{B8 - B4}{B8 + B4}$$

El resultado es un valor normalizado entre -1 y 1:
* **Valores < 0.1:** Cuerpos de agua, nubes, o concreto/asfalto puro (Zonas críticas de contaminación).
* **Valores 0.2 - 0.4:** Arbustos, pastizales o zonas urbanas mixtas.
* **Valores > 0.6:** Bosques densos y cultivos saludables (Cañaduzales al norte o Farallones al oeste).

In [3]:
# 1. Calcular el NDVI usando la matemática de tensores de Earth Engine
# GEE tiene una función optimizada para hacer esta división de bandas píxel por píxel
ndvi_cali = s2_mosaico.normalizedDifference(['B8', 'B4']).rename('NDVI')

# 2. Configurar una paleta científica intuitiva
# Rojo/Marrón para ciudad/asfalto, Amarillo para transición, Verde oscuro para bosque
viz_ndvi = {
    'min': 0.0,
    'max': 0.8,
    'palette': ['#d73027', '#f46d43', '#fdae61', '#fee08b', '#d9ef8b', '#a6d96a', '#66bd63', '#1a9850']
}

# En lugar de usar el 'Map' anterior, creamos un 'Map2' exclusivo para el NDVI
Map2 = geemap.Map(basemap='SATELLITE')
Map2.centerObject(cali_roi, 12)

# Añadimos la capa al nuevo mapa
Map2.addLayer(ndvi_cali, viz_ndvi, 'NDVI (Densidad de Vegetación)')
Map2.add_colorbar(viz_ndvi, label="NDVI (Asfalto/Suelo desnudo ➔ Vegetación densa)", layer_name="NDVI")
Map2.addLayerControl()

Map2

Map(center=[3.424999763692953, -76.50000000000118], controls=(WidgetControl(options=['position', 'transparent_…